# Data Augmentation for Text-to-SQL
## Experimental Notebook — BSc Thesis

**Author:** E. W. S. Anuradha  
**Institution:** [Your University Name]  
**Year:** 2026  

**Thesis title:** *Data Augmentation for Enhancing SQL Query Generation: A Natural Language Processing Approach*

---

This notebook implements and evaluates five experimental stages of schema-aware
data augmentation applied to a T5-base model fine-tuned on the Spider benchmark
using NatSQL as the intermediate representation.

| Stage | Configuration | Purpose |
|-------|--------------|---------|
| 1 | Baseline | No augmentation — raw NatSQL input |
| 2 | Tok+Comp | Token preprocessing + compositional boundary markers |
| 3 | Tok+Comp + ContextTok | + Schema-aware contextual token splitting |
| 4 | Tok+Comp + AliasNorm | + Alias normalization (no ContextTok) |
| 5 | Tok+Comp + ContextTok + AliasNorm | Full proposed pipeline |

## Setup: Spider Dataset and NatSQL Pipeline Preparation

This cell handles one-time setup: downloading the Spider dataset via the Kaggle
API, cloning the NatSQL repository, and converting Spider's original SQL
annotations into NatSQL format.

**Steps performed:**
- Installs the Kaggle CLI and authenticates using the API key
- Downloads and unzips the Spider 1.0 dataset from Kaggle
- Clones the [NatSQL repository](https://github.com/ygan/NatSQL) and copies
  Spider's `train_spider.json`, `dev.json`, `tables.json`, and `database/` into it
- Applies three compatibility patches to `TokenString.py` to fix runtime errors
  in the NatSQL preprocessing pipeline (tuple concatenation, deprecated `LEMMA`
  usage, and a typo in `table_transform.py`)
- Runs `check_and_preprocess.sh` to generate NatSQL-converted training and
  validation files
- Copies the generated `train_spider_natsql.json` and `dev_natsql.json` back
  into the Spider data directory for use in subsequent cells

*Note: This cell only needs to be run once per Colab session. The patches are
applied programmatically and do not modify the original repository permanently.*

In [ ]:
!pip install -q kaggle

import json, os

token = "KGAT_f499bc26dfa7032eac973a6314415941"
kaggle_username = "ShalithaChamp"

kaggle_json = {
    "username": kaggle_username,
    "key": token,
}

os.makedirs("/root/.kaggle", exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_json, f)

!chmod 600 /root/.kaggle/kaggle.json

print("✅ kaggle.json created from hard-coded username + key!")

!kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset
!unzip -q yale-universitys-spider-10-nlp-dataset.zip -d spider_data


# ---------------------------------------------Add NatSQL with below codes --- Start
!git clone https://github.com/ygan/NatSQL.git
%cd NatSQL
!ls


%cd /content/NatSQL
!mkdir -p data

!cp /content/spider_data/spider/train_spider.json data/train_spider.json
!cp /content/spider_data/spider/dev.json          data/dev.json
!cp /content/spider_data/spider/tables.json       data/tables.json
!cp -r /content/spider_data/spider/database       data/database

!ls data




%cd /content/NatSQL

# Patch TokenString.py to fix TypeError (tuple + tuple vs list)
import io, re

path = "natsql2sql/preprocess/TokenString.py"

with open(path, "r") as f:
    txt = f.read()

old = "suffixes = nlp.Defaults.suffixes +  (r'((\\d{4}((_|-|/){1}\\d{2}){2})|((\\d{2})(_|-|/)){2}\\d{4})(\\s\\d{2}(:\\d{2}){2}){0,1}',) + (r'(\\d{1,2}(st|nd|rd|th){0,1}(,|\\s)){0,1}((J|j)an(uary){0,1}|(F|f)eb(ruary){0,1}|(M|m)ar(ch){0,1}|(A|a)pr(il){0,1}|(M|m)ay|(J|j)un(e){0,1}|(J|j)ul(y){0,1}|(A|a)ug(ust){0,1}|(S|s)ep(tember){0,1}|(O|o)ct(ober){0,1}|(N|n)ov(ember){0,1}|(D|d)ec(ember){0,1})(\\s|,)(\\d{1,2}(st|nd|rd|th){0,1}(\\s|,){1,3}){0,1}\\d{4}',) + ( r'(\\d{1,6}(_|-|\\+|/)\\d{0,6}[A-Za-z]{0,6}\\d{0,6}[A-Za-z]{0,6})',)"
new = "suffixes = list(nlp.Defaults.suffixes) + [r'((\\d{4}((_|-|/){1}\\d{2}){2})|((\\d{2})(_|-|/)){2}\\d{4})(\\s\\d{2}(:\\d{2}){2}){0,1}', r'(\\d{1,2}(st|nd|rd|th){0,1}(,|\\s)){0,1}((J|j)an(uary){0,1}|(F|f)eb(ruary){0,1}|(M|m)ar(ch){0,1}|(A|a)pr(il){0,1}|(M|m)ay|(J|j)un(e){0,1}|(J|j)ul(y){0,1}|(A|a)ug(ust){0,1}|(S|s)ep(tember){0,1}|(O|o)ct(ober){0,1}|(N|n)ov(ember){0,1}|(D|d)ec(ember){0,1})(\\s|,)(\\d{1,2}(st|nd|rd|th){0,1}(\\s|,){1,3}){0,1}\\d{4}', r'(\\d{1,6}(_|-|\\+|/)\\d{0,6}[A-Za-z]{0,6}\\d{0,6}[A-Za-z]{0,6})']"

if old in txt:
    txt = txt.replace(old, new)
else:
    # simpler generic patch: wrap suffixes in list()
    txt = txt.replace("suffixes = nlp.Defaults.suffixes +", "suffixes = list(nlp.Defaults.suffixes) +")

with open(path, "w") as f:
    f.write(txt)

print("✅ Patched TokenString.py")

%cd /content/NatSQL

path = "natsql2sql/preprocess/TokenString.py"

with open(path, "r") as f:
    txt = f.read()

# Replace the problematic special_case with a simpler ORTH-only version
txt = txt.replace(
    "nlp.tokenizer.add_special_case(u'Ph.D', [{ORTH: u'Ph.D', LEMMA: u'ph.d'}])",
    "nlp.tokenizer.add_special_case(u'Ph.D', [{ORTH: u'Ph.D'}])"
)

with open(path, "w") as f:
    f.write(txt)

print("✅ Patched Ph.D special case in TokenString.py")

%cd /content/NatSQL

path = "natsql2sql/preprocess/TokenString.py"
with open(path, "r") as f:
    txt = f.read()

# 1) Make sure suffixes use list(...) (in case it wasn't generic‑patched yet)
txt = txt.replace(
    "suffixes = nlp.Defaults.suffixes +",
    "suffixes = list(nlp.Defaults.suffixes) +"
)

# 2) Remove all LEMMA usages from tokenizer special cases
#    e.g. [{ORTH: u'Ph.D', LEMMA: u'ph.d'}] -> [{ORTH: u'Ph.D'}]
import re
txt = re.sub(r"\{ORTH:\s*u'([^']+?)',\s*LEMMA:\s*u'[^']*?'\}", r"{ORTH: u'\1'}", txt)
txt = re.sub(r'\{ORTH:\s*u"([^"]+?)",\s*LEMMA:\s*u"[^"]*?"\}', r'{ORTH: u"\1"}', txt)

with open(path, "w") as f:
    f.write(txt)

print("✅ Patched TokenString.py (suffixes + removed all LEMMA in special cases)")

%cd /content/NatSQL

path = "table_transform.py"
with open(path, "r") as f:
    txt = f.read()

# Fix the typo: "STOP_WORDSand" -> "STOP_WORDS and"
txt = txt.replace("STOP_WORDSand and", "STOP_WORDS and")

with open(path, "w") as f:
    f.write(txt)

print("✅ Patched table_transform.py (STOP_WORDSand typo)")



!bash check_and_preprocess.sh
!ls NatSQLv1_6


!cp /content/NatSQL/NatSQLv1_6/train_spider.json /content/spider_data/spider/train_spider_natsql.json
!cp /content/NatSQL/NatSQLv1_6/dev.json          /content/spider_data/spider/dev_natsql.json

!ls /content/spider_data/spider


# ---------------------------------------------Add NatSQL with below codes --- Finish


✅ kaggle.json created from hard-coded username + key!
Dataset URL: https://www.kaggle.com/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset
License(s): unknown
User cancelled operation
unzip:  cannot find or open yale-universitys-spider-10-nlp-dataset.zip, yale-universitys-spider-10-nlp-dataset.zip.zip or yale-universitys-spider-10-nlp-dataset.zip.ZIP.
Cloning into 'NatSQL'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (50/50), done.
^C
[Errno 2] No such file or directory: 'NatSQL'
/content/NatSQL
check_and_preprocess.sh			 pattern_generation.py
config.py				 punkt.py
data					 __pycache__
generate_spider_examples_with_natsql.py  README.md
natsql2sql				 requirements.txt
natsql2sql.sh				 run.py
natsql2sql_without_values.sh		 setence_split.py
NatSQLv1_6				 spider_natsql_t5base_full
NatSQLv1_6_1				 table_transform.py
/content/NatSQL
20k-original.pkl  conceptnet.pkl  dev.json     train_spider.json


## Section 1: Environment Setup and Data Loading

This cell sets up all dependencies and prepares the data and schema structures
needed by all subsequent cells. It must be run first.

**What is loaded:**
- `train_spider.json` / `dev.json` — Spider NL questions and gold SQL (7,000 train, 1,034 dev)
- `train_spider_natsql.json` / `dev_natsql.json` — NatSQL-aligned labels for the same splits
- `tables.json` — Spider schema file used to build the `SCHEMAS` lookup dictionary

**Key outputs produced:**
- `train_ds` / `val_ds` — HuggingFace `Dataset` objects (shuffled, seed 42)
- `SCHEMAS` — index of `db_id → {tables, table_to_cols}` used by ContextTok and AliasNorm
- `tokenizer` — `T5TokenizerFast` loaded from `t5-base`

*Note: The `context` field is set to `db_id` at this stage. Richer schema context
strings are derived from `SCHEMAS` inside each stage preprocessor in Section 2.*

In [ ]:
# Cell 1: Load Spider+NatSQL, Spider schema, and prepare t5-base tokenizer
!pip install -q transformers datasets accelerate

import os, json, re, torch
from datasets import Dataset
from transformers import T5TokenizerFast, T5ForConditionalGeneration, Trainer, TrainingArguments

BASE_SPIDER = "/content/spider_data/spider"
BASE_NATSQL = BASE_SPIDER

# -------------------------
# Load Spider examples
# -------------------------
with open(os.path.join(BASE_SPIDER, "train_spider.json")) as f:
    spider_train = json.load(f)
with open(os.path.join(BASE_SPIDER, "dev.json")) as f:
    spider_dev = json.load(f)

# NatSQL-aligned
with open(os.path.join(BASE_NATSQL, "train_spider_natsql.json")) as f:
    natsql_train = json.load(f)
with open(os.path.join(BASE_NATSQL, "dev_natsql.json")) as f:
    natsql_dev = json.load(f)

assert len(spider_train) == len(natsql_train)
assert len(spider_dev) == len(natsql_dev)

def build_split(spider_exs, natsql_exs):
    qs, sqls, nss, dbs, ctx = [], [], [], [], []
    for ex_sp, ex_ns in zip(spider_exs, natsql_exs):
        qs.append(ex_sp["question"])
        sqls.append(ex_sp["query"])
        nss.append(ex_ns["NatSQL"])
        dbs.append(ex_sp["db_id"])
        # simple context = db_id (we derive richer schema context from SCHEMAS later)
        ctx.append(ex_sp["db_id"])
    return {"question": qs, "sql": sqls, "natsql": nss, "db_id": dbs, "context": ctx}

train_dict = build_split(spider_train, natsql_train)
dev_dict   = build_split(spider_dev,   natsql_dev)

train_ds = Dataset.from_dict(train_dict)
val_ds   = Dataset.from_dict(dev_dict)

# BEST MODE: Full dataset
train_ds = train_ds.shuffle(seed=42)
val_ds   = val_ds.shuffle(seed=42)

print("Train:", len(train_ds), "Val:", len(val_ds))
print("Sample Q:", train_ds[0]["question"])
print("Sample NatSQL:", train_ds[0]["natsql"])

# -------------------------
# Load Spider schema (tables.json)
# -------------------------
tables_path = os.path.join(BASE_SPIDER, "tables.json")
with open(tables_path) as f:
    tables_json = json.load(f)

# Build a schema index: db_id -> {table_name: set(columns)}
# Using cleaned names (table_names, column_names) as in Spider.
SCHEMAS = {}  # db_id -> {"tables": set(str), "table_to_cols": {table: set(columns)}}

for db in tables_json:
    db_id = db["db_id"]
    table_names = db["table_names"]           # cleaned table names
    column_names = db["column_names"]         # list of [table_idx, column_name]
    table_to_cols = {t: set() for t in table_names}

    for (t_idx, col_name) in column_names:
        if t_idx == -1:
            # -1 means "*", not a real table
            continue
        table = table_names[t_idx]
        table_to_cols[table].add(col_name)

    SCHEMAS[db_id] = {
        "tables": set(table_names),
        "table_to_cols": table_to_cols,
    }

print("Schema example for db:", spider_train[0]["db_id"])
print(SCHEMAS[spider_train[0]["db_id"]])

base_model_name = "t5-base"
tokenizer = T5TokenizerFast.from_pretrained(base_model_name)

print("✅ Cell 1: Spider+NatSQL+schema loaded, t5-base tokenizer ready.")


Train: 7000 Val: 1034
Sample Q: What is the average enrollment of schools?
Sample NatSQL: select avg ( school.Enrollment )  from school 
Schema example for db: department_management
{'tables': {'head', 'department', 'management'}, 'table_to_cols': {'department': {'ranking', 'name', 'department id', 'budget in billions', 'creation', 'num employees'}, 'head': {'born state', 'age', 'name', 'head id'}, 'management': {'temporary acting', 'department id', 'head id'}}}
✅ Cell 1: Spider+NatSQL+schema loaded, t5-base tokenizer ready.


In [ ]:
# CELL 2 — Stage-specific preprocessing + training

import re
from copy import deepcopy

# ---------- Original paper-style helpers (Tok+Comp) ----------

def paper_token_preprocessing(text: str) -> str:
    text = re.sub(r'_', ' _ ', text)
    text = re.sub(r'([a-z0-9])([A-Z])', r'\1 _ \2', text)
    text = re.sub(r'\.', ' . ', text)
    kw_map = {"avg": "average", "max": "maximum", "min": "minimum", "cnt": "count"}
    for k, v in kw_map.items():
        text = re.sub(rf"\b{k}\b", v, text, flags=re.I)
    return text

def spider_ss_boundary(text: str) -> str:
    s = text
    s = re.sub(r"\bselect\b", "[sep1] select", s, flags=re.I)
    if re.search(r"\bwhere\b", s, flags=re.I):
        s = re.sub(r"\bwhere\b", "[/sep1] where [sep2]", s, flags=re.I)
        s = s + " [/sep2]"
    else:
        s = s + " [/sep1]"
    return s

# ---------- Schema-aware contextual token splitting ----------

CAMEL_CASE_RE = re.compile(r'([a-z0-9])([A-Z])')

SQL_KEYWORD_MAP = {
    "avg": "average",
    "max": "maximum",
    "min": "minimum",
    "cnt": "count",
    "asc": "ascending",
    "desc": "descending",
}

def normalize_identifier(name: str) -> str:
    s = name
    s = re.sub(r'_', ' ', s)
    s = CAMEL_CASE_RE.sub(r'\1 \2', s)
    parts = s.split()
    new_parts = []
    for p in parts:
        low = p.lower()
        if low in SQL_KEYWORD_MAP:
            new_parts.append(SQL_KEYWORD_MAP[low])
        else:
            new_parts.append(p)
    return " ".join(new_parts)

def schema_has_table_and_column(db_id: str, table: str, col: str) -> bool:
    """Check if table and column exist in Spider schema for db_id. """
    schema = SCHEMAS.get(db_id)
    if not schema:
        return False
    t_norm = table.lower()
    c_norm = col.lower()
    for t in schema["tables"]:
        if t.lower() == t_norm:
            for c in schema["table_to_cols"][t]:
                if c.lower() == c_norm:
                    return True
    return False

def contextual_split_token(token: str, db_id: str) -> str:
    """
    Schema-aware token splitting:
    - If token looks like Table.Column and (Table, Column) is in schema: keep as Table.Column.
    - Else, do NOT force split; only normalize identifier (snake/camel/keywords).
    """
    stripped = token.strip()
    m = re.match(r'^([A-Za-z0-9_]+)\.([A-Za-z0-9_]+)$', stripped)
    if m:
        t, c = m.group(1), m.group(2)
        if schema_has_table_and_column(db_id, t, c):
            return f"{t}.{c}"
        # fall through if not a valid pair

    base = re.sub(r'[^\w]', '', stripped)
    if not base:
        return token
    norm = normalize_identifier(base)
    if norm != base:
        return norm
    return token

def preprocess_contextual_tokens_text(text: str, db_id: str) -> str:
    """
    Apply contextual token splitting to arbitrary text using db-specific schema.
    """
    tokens = text.split()
    norm_tokens = [contextual_split_token(t, db_id) for t in tokens]
    return " ".join(norm_tokens)

# ---------- Schema-aware alias normalization (input-side only) ----------

def extract_alias_map(sql_like: str, db_id: str):
    """
    Build alias -> table map by:
    - detecting FROM table alias / JOIN table alias patterns,
    - ensuring 'table' is a real table in the schema.
    """
    schema = SCHEMAS.get(db_id)
    if not schema:
        return {}

    alias_map = {}
    pattern = re.compile(
        r'\b(from|join)\s+([a-zA-Z0-9_]+)\s+(as\s+)?([a-zA-Z0-9_]+)\b',
        re.IGNORECASE,
    )

    for m in pattern.finditer(sql_like):
        table = m.group(2)
        alias = m.group(4)
        if any(table.lower() == t.lower() for t in schema["tables"]):
            alias_map[alias] = table
    return alias_map

def apply_alias_normalization_in_text(text: str, db_id: str) -> str:
    """
    Expand aliases in a SQL-like string:
    - Replace alias.Column with Table.Column when alias->Table is known.
    - Used only on INPUT; labels stay as original NatSQL.
    """
    alias_map = extract_alias_map(text, db_id)
    if not alias_map:
        return text

    result = text
    for alias, table in alias_map.items():
        result = re.sub(
            r'\b' + re.escape(alias) + r'\.([A-Za-z0-9_]+)',
            lambda m: f"{table}.{m.group(1)}",
            result,
        )
    return result

# ---------- Stage-specific preprocessors ----------

# Stage 1) Baseline: simple input, labels = raw NatSQL
def preprocess_baseline(batch):
    inputs = [f"db: {c} question: {q}" for c, q in zip(batch["context"], batch["question"])]
    model_inputs = tokenizer(inputs, max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(batch["natsql"], max_length=256, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Stage 2) Tok+Comp (paper): input = Tok+Comp-preprocessed context, labels = raw NatSQL
def preprocess_tokcomp(batch):
    inputs_text = []
    for q, c, db_id in zip(batch["question"], batch["context"], batch["db_id"]):
        c1 = paper_token_preprocessing(c)
        c2 = spider_ss_boundary(c1)
        inputs_text.append(f"db: {c2} question: {q}")
    model_inputs = tokenizer(inputs_text, max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(batch["natsql"], max_length=256, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Stage 3) Tok+Comp + ContextTok:
# take Tok+Comp-style schema string, then apply schema-aware contextual splitting on top (input only).
def preprocess_tokcomp_contexttok(batch):
    inputs_text = []
    for q, c, db_id in zip(batch["question"], batch["context"], batch["db_id"]):
        # derive a simple schema string (tables + columns) as context
        schema = SCHEMAS.get(db_id, {"table_to_cols": {}})
        schema_str_parts = []
        for t, cols in schema["table_to_cols"].items():
            schema_str_parts.append(t)
            for col in cols:
                schema_str_parts.append(f"{t}.{col}")
        schema_str = " ".join(schema_str_parts) if schema_str_parts else c

        # Tok+Comp on that schema string
        tok = paper_token_preprocessing(schema_str)
        comp = spider_ss_boundary(tok)

        # Contextual token splitting
        ctx_processed = preprocess_contextual_tokens_text(comp, db_id)

        inputs_text.append(f"db: {ctx_processed} question: {q}")

    model_inputs = tokenizer(inputs_text, max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(batch["natsql"], max_length=256, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Stage 4) Tok+Comp + AliasNorm:
# use NatSQL to derive alias map, expand aliases in a SQL-like context, then Tok+Comp (no ContextTok).
def preprocess_tokcomp_aliasnorm(batch):
    inputs_text = []
    for q, c, db_id, ns in zip(batch["question"], batch["context"], batch["db_id"], batch["natsql"]):
        # build SQL-like string from NatSQL then apply alias normalization
        alias_expanded = apply_alias_normalization_in_text(ns, db_id)

        # Tok+Comp on alias-expanded string
        tok = paper_token_preprocessing(alias_expanded)
        comp = spider_ss_boundary(tok)

        inputs_text.append(f"db: {comp} question: {q}")

    model_inputs = tokenizer(inputs_text, max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(batch["natsql"], max_length=256, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Stage 5) Tok+Comp + ContextTok + AliasNorm (full model):
# alias normalization -> Tok+Comp -> schema-aware contextual splitting (input only).
def preprocess_tokcomp_contexttok_aliasnorm(batch):
    inputs_text = []
    for q, c, db_id, ns in zip(batch["question"], batch["context"], batch["db_id"], batch["natsql"]):
        # 1) alias normalization on NatSQL
        alias_expanded = apply_alias_normalization_in_text(ns, db_id)

        # 2) Tok+Comp on alias-expanded string
        tok = paper_token_preprocessing(alias_expanded)
        comp = spider_ss_boundary(tok)

        # 3) schema-aware contextual splitting on top
        ctx_processed = preprocess_contextual_tokens_text(comp, db_id)

        inputs_text.append(f"db: {ctx_processed} question: {q}")

    model_inputs = tokenizer(inputs_text, max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(batch["natsql"], max_length=256, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# ---------- Stage registry (comment any you want to disable) ----------

preprocessors = {
    "1.Baseline":                           preprocess_baseline,
    "2.Tok+Comp":                           preprocess_tokcomp,
    "3.Tok+Comp+ContextTok":                preprocess_tokcomp_contexttok,
    "4.Tok+Comp+AliasNorm":                 preprocess_tokcomp_aliasnorm,
    "5.Tok+Comp+ContextTok+AliasNorm":      preprocess_tokcomp_contexttok_aliasnorm,
}

training_args = TrainingArguments(
    output_dir="./spider_natsql_t5base_full",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=3e-4,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to=[],
)

trained_models = {}

for stage_name, preprocess_fn in preprocessors.items():
    print("\n" + "="*60)
    print(f"🚀 TRAINING STAGE: {stage_name}")
    print("="*60)

    train_proc = train_ds.map(
        preprocess_fn,
        batched=True,
        remove_columns=train_ds.column_names,
    )
    train_proc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    model = T5ForConditionalGeneration.from_pretrained(base_model_name)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_proc,
        eval_dataset=None,
    )

    trainer.train()
    trained_models[stage_name] = (model, preprocess_fn)
    print(f"✅ Finished training: {stage_name}")

    del trainer
    torch.cuda.empty_cache()

print("✅ Cell 2: all enabled stages trained on NatSQL with schema-aware preprocessing.")



🚀 TRAINING STAGE: 1.Baseline


Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Step,Training Loss
100,1.207168
200,0.094954
300,0.086096
400,0.087956
500,0.084895
600,0.079216
700,0.077423
800,0.079742
900,0.078652
1000,0.079393


✅ Finished training: 1.Baseline

🚀 TRAINING STAGE: 2.Tok+Comp


...

✅ Finished training: 4.Tok+Comp+AliasNorm

🚀 TRAINING STAGE: 5.Tok+Comp+ContextTok+AliasNorm


Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Step,Training Loss
100,0.715097
200,0.003760
300,0.002307
400,0.001953
500,0.002109
600,0.001429
700,0.001611
800,0.000922
900,0.000959
1000,0.000586


✅ Finished training: 5.Tok+Comp+ContextTok+AliasNorm
✅ Cell 2: all enabled stages trained on NatSQL with schema-aware preprocessing.


In [ ]:
# CELL 3 — Evaluate NatSQL EM + token-level F1

from tqdm.auto import tqdm

def generate_natsql_for_stage(stage_name, model, preprocess_fn):
    print(f"\n=== Generating NatSQL for stage: {stage_name} ===")
    val_proc = val_ds.map(
        preprocess_fn,
        batched=True,
        remove_columns=val_ds.column_names,
    )
    val_proc.set_format(type="torch", columns=["input_ids", "attention_mask"])

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    preds = []
    batch_size = 32
    for i in tqdm(range(0, len(val_proc), batch_size)):
        batch = val_proc[i : i + batch_size]
        batch_in = {
            "input_ids": batch["input_ids"].to(device),
            "attention_mask": batch["attention_mask"].to(device),
        }
        with torch.no_grad():
            out = model.generate(
                **batch_in,
                max_length=256,
                num_beams=1,
            )
        batch_txt = tokenizer.batch_decode(out, skip_special_tokens=True)
        preds.extend(batch_txt)
    return preds

def normalize_natsql(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r'\s+', ' ', s)
    return s

def compute_em_f1(preds, golds):
    em = 0
    total_f1 = 0.0
    for p, g in zip(preds, golds):
        p_tok = normalize_natsql(p).split()
        g_tok = normalize_natsql(g).split()

        if p_tok == g_tok:
            em += 1

        common = set(p_tok) & set(g_tok)
        if p_tok and g_tok and common:
            prec = len(common) / len(p_tok)
            rec  = len(common) / len(g_tok)
            f1 = 2 * prec * rec / (prec + rec)
        else:
            f1 = 0.0
        total_f1 += f1

    em_pct = em / len(golds) * 100.0
    f1_pct = total_f1 / len(golds) * 100.0
    return em_pct, f1_pct

gold_natsql = val_ds["natsql"]

stage_metrics = {}

ordered_all = [
    "1.Baseline",
    "2.Tok+Comp",
    "3.Tok+Comp+ContextTok",
    "4.Tok+Comp+AliasNorm",
    "5.Tok+Comp+ContextTok+AliasNorm",
]
ordered_stages = [name for name in ordered_all if name in trained_models]

for stage_name in ordered_stages:
    model, preprocess_fn = trained_models[stage_name]
    preds = generate_natsql_for_stage(stage_name, model, preprocess_fn)

    print(f"\n=== DEBUG EXAMPLES for {stage_name} ===")
    for i in range(3):
        print("Q   :", val_ds[i]["question"])
        print("GOLD:", gold_natsql[i])
        print("PRED:", preds[i])
        print("-----")

    em, f1 = compute_em_f1(preds, gold_natsql)
    stage_metrics[stage_name] = {"EM": em, "F1": f1}
    print(f"✅ {stage_name} -> NatSQL EM: {em:.2f}% | Token F1: {f1:.2f}%")

print("\n" + "="*80)
print(f"{'Stage':<34} {'EM %':>8} {'F1 %':>8} {'ΔEM vs Base':>14}")
print("-"*80)

if "1.Baseline" in stage_metrics:
    baseline_em = stage_metrics["1.Baseline"]["EM"]
else:
    first_name = ordered_stages[0]
    baseline_em = stage_metrics[first_name]["EM"]

for name in ordered_stages:
    em = stage_metrics[name]["EM"]
    f1 = stage_metrics[name]["F1"]
    if "1.Baseline" in stage_metrics:
        d_base = f"{em - baseline_em:+.2f}" if name != "1.Baseline" else "-"
    else:
        d_base = "-" if name == ordered_stages[0] else f"{em - baseline_em:+.2f}"
    print(f"{name:<34} {em:>7.2f}% {f1:>7.2f}% {d_base:>14}")

print("="*80)



=== Generating NatSQL for stage: 1.Baseline ===


Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

  0%|          | 0/33 [00:00<?, ?it/s]


=== DEBUG EXAMPLES for 1.Baseline ===
Q   : How many players are from each country?
GOLD: select count ( players.* )  , players.country_code from players  group by players.country_code 
PRED: select count ( player.* ) from player where player.Country = "%%"
-----
Q   : How many documents are using the template with type code 'PPT'?
GOLD: select count ( documents.* )  from documents  where  templates.Template_Type_Code = "PPT" 
PRED: select count ( documents.* ) from documents where documents.Template_Type_Code = "PPT"
-----
Q   : How many cartoons were written by "Joseph Kuhr"?
GOLD: select count ( cartoon.* )  from cartoon  where  cartoon.Written_by = "Joseph Kuhr" 
PRED: select count ( cartoon.* ) from cartoon where cartoon.Name = "Joseph Kuhr"
-----
✅ 1.Baseline -> NatSQL EM: 6.09% | Token F1: 63.42%

=== Generating NatSQL for stage: 2.Tok+Comp ===


Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

...

  0%|          | 0/33 [00:00<?, ?it/s]


=== DEBUG EXAMPLES for 4.Tok+Comp+AliasNorm ===
Q   : How many players are from each country?
GOLD: select count ( players.* )  , players.country_code from players  group by players.country_code 
PRED: select count ( players.* ) , players.country_code from players group by players.country_code
-----
Q   : How many documents are using the template with type code 'PPT'?
GOLD: select count ( documents.* )  from documents  where  templates.Template_Type_Code = "PPT" 
PRED: select count ( documents.* ) from documents where templates.Template_Type_Code = "PPT"
-----
Q   : How many cartoons were written by "Joseph Kuhr"?
GOLD: select count ( cartoon.* )  from cartoon  where  cartoon.Written_by = "Joseph Kuhr" 
PRED: select count ( cartoon.* ) from cartoon where cartoon.Written_by = "Joseph Kuhr"
-----
✅ 4.Tok+Comp+AliasNorm -> NatSQL EM: 77.76% | Token F1: 92.79%

=== Generating NatSQL for stage: 5.Tok+Comp+ContextTok+AliasNorm ===


Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

  0%|          | 0/33 [00:00<?, ?it/s]


=== DEBUG EXAMPLES for 5.Tok+Comp+ContextTok+AliasNorm ===
Q   : How many players are from each country?
GOLD: select count ( players.* )  , players.country_code from players  group by players.country_code 
PRED: select count ( players.* ) , players.country_code from players group by players.country_code
-----
Q   : How many documents are using the template with type code 'PPT'?
GOLD: select count ( documents.* )  from documents  where  templates.Template_Type_Code = "PPT" 
PRED: select count ( documents.* ) from documents where templates.Template_Type_Code = "PPT"
-----
Q   : How many cartoons were written by "Joseph Kuhr"?
GOLD: select count ( cartoon.* )  from cartoon  where  cartoon.Written_by = "Joseph Kuhr" 
PRED: select count ( cartoon.* ) from cartoon where cartoon.Writed_by = "Joseph Kuhr"
-----
✅ 5.Tok+Comp+ContextTok+AliasNorm -> NatSQL EM: 77.27% | Token F1: 92.50%

Stage                                  EM %     F1 %    ΔEM vs Base
----------------------------------------